# 🏥 Prédiction de l'Anémie - Notebook Final

## 1. Contexte et Objectifs
L'anémie est une condition dans laquelle vous manquez de suffisamment de globules rouges sains pour transporter un oxygène adéquat vers les tissus de votre corps. Ce projet vise à développer un modèle de Machine Learning capable de prédire la présence d'anémie chez un patient en se basant sur des mesures hématologiques standard.

### Objectifs :
- Analyse exploratoire des données (EDA)
- Prétraitement et ingénierie des caractéristiques
- Comparaison de plusieurs modèles de classification
- Sélection et évaluation du meilleur modèle
- Export du modèle pour une mise en production

## 2. Import des bibliothèques et Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
print('✅ Bibliothèques importées')

In [ ]:
# Chargement du dataset
df = pd.read_csv('../data/raw/anemia.csv')
print(f"Dimensions du dataset : {df.shape}")
df.head()

## 3. Analyse Exploratoire des Données (EDA)
Nous analysons la distribution des variables et leur relation avec la cible (Result).

In [ ]:
# Distribution de la variable cible
plt.figure(figsize=(8, 5))
sns.countplot(x='Result', data=df, palette='viridis')
plt.title('Distribution de la présence d\'Anémie (0: Non, 1: Oui)')
plt.show()

# Analyse des variables numériques par rapport à la cible
numeric_features = ['Hemoglobin', 'MCH', 'MCHC', 'MCV']
plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(x='Result', y=col, data=df)
    plt.title(f'Distribution de {col} vs Result')
plt.tight_layout()
plt.show()

## 4. Prétraitement des données
Nous allons traiter les outliers et préparer les données pour l'entraînement.

In [ ]:
# Traitement des outliers par capping
for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

print("✅ Outliers traités")

In [ ]:
# Division des données
X = df.drop('Result', axis=1)
y = df['Result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille d'entraînement : {X_train.shape}")
print(f"Taille de test : {X_test.shape}")

## 5. Modélisation et Comparaison
Nous testons plusieurs algorithmes pour trouver le plus performant.

In [ ]:
# Définition du préprocesseur (Scaling)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features)
], remainder='passthrough')

# Liste des modèles à tester
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(random_state=42, probability=True),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = []

for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    results.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
    })

df_results = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
display(df_results)

## 6. Évaluation Finale du Meilleur Modèle

In [ ]:
best_model_name = df_results.iloc[0]['Modèle']
print(f"🏆 Meilleur modèle sélectionné : {best_model_name}")

best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', models[best_model_name])
])

best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

print("\nClassification Report :")
print(classification_report(y_test, y_pred))

# Matrice de confusion
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title(f'Matrice de Confusion - {best_model_name}')
plt.show()

## 7. Sauvegarde du modèle
On sauvegarde le pipeline complet pour l'utiliser dans l'application Streamlit.

In [ ]:
joblib.dump(best_pipeline, '../models/best_model.pkl')
print("✅ Modèle sauvegardé dans models/best_model.pkl")

## 8. Conclusion
Le modèle est capable de prédire l'anémie avec une excellente précision. Ce notebook a permis de comparer plusieurs approches et de sélectionner la plus robuste pour la mise en production.